# 🎬 Movie Recommendation System — EDA & Model Walkthrough

This notebook walks through:
1. Data loading & exploration
2. Feature engineering
3. Content-based filtering (Cosine Similarity)
4. Collaborative filtering (SVD)
5. Evaluation

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import ast
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

plt.style.use('dark_background')
pd.set_option('display.max_columns', None)
print('✅ Imports OK')

## 2. Load Data

In [ ]:
movies = pd.read_csv('../data/tmdb_5000_movies.csv')
credits = pd.read_csv('../data/tmdb_5000_credits.csv')

print('Movies shape:', movies.shape)
print('Credits shape:', credits.shape)
movies.head(3)

## 3. Exploratory Data Analysis

In [ ]:
# Missing values
print('Missing values:\n', movies.isnull().sum()[movies.isnull().sum() > 0])

In [ ]:
# Rating distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(movies['vote_average'], bins=30, color='#e8b84b', edgecolor='black')
axes[0].set_title('Vote Average Distribution')
axes[0].set_xlabel('Rating')

axes[1].hist(movies['vote_count'].clip(0, 5000), bins=40, color='#e05c5c', edgecolor='black')
axes[1].set_title('Vote Count Distribution (clipped at 5000)')
axes[1].set_xlabel('Vote Count')

plt.tight_layout()
plt.show()

In [ ]:
# Top genres
from collections import Counter

all_genres = []
for row in movies['genres']:
    try:
        for g in ast.literal_eval(row):
            all_genres.append(g['name'])
    except:
        pass

genre_counts = Counter(all_genres).most_common(15)
labels, values = zip(*genre_counts)

plt.figure(figsize=(12, 5))
plt.barh(labels, values, color='#e8b84b')
plt.title('Top 15 Genres')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 4. Feature Engineering — Content-Based

In [ ]:
# Merge
df = movies.merge(credits, on='title')

def parse_list(text, key='name', limit=None):
    try:
        items = ast.literal_eval(text)
        names = [i[key].replace(' ', '') for i in items]
        return names[:limit] if limit else names
    except:
        return []

def get_director(crew_text):
    try:
        for m in ast.literal_eval(crew_text):
            if m.get('job') == 'Director':
                return m['name'].replace(' ', '')
    except:
        pass
    return ''

df['genres']   = df['genres'].apply(lambda x: parse_list(x))
df['keywords'] = df['keywords'].apply(lambda x: parse_list(x))
df['cast']     = df['cast'].apply(lambda x: parse_list(x, limit=3))
df['director'] = df['crew'].apply(get_director)
df['overview'] = df['overview'].fillna('').apply(str.split)

# Combined tag soup
df['tags'] = (
    df['overview'] + df['genres'] + df['keywords'] +
    df['cast'] + df['director'].apply(lambda x: [x] if x else [])
)
df['tags'] = df['tags'].apply(lambda x: ' '.join(x).lower())

df[['title', 'tags']].head(3)

## 5. Content-Based Filtering

In [ ]:
cv = CountVectorizer(max_features=5000, stop_words='english')
vectors = cv.fit_transform(df['tags']).toarray()
similarity = cosine_similarity(vectors)

print(f'Similarity matrix shape: {similarity.shape}')

def recommend(title, n=5):
    title_index = {t: i for i, t in enumerate(df['title'])}
    if title not in title_index:
        print(f'Movie not found: {title}')
        return
    idx = title_index[title]
    distances = sorted(enumerate(similarity[idx]), key=lambda x: x[1], reverse=True)
    recs = [df.iloc[i]['title'] for i, _ in distances[1:n+1]]
    return recs

# Test it!
print('\nRecommendations for "The Dark Knight":')
for i, r in enumerate(recommend('The Dark Knight'), 1):
    print(f'  {i}. {r}')

In [ ]:
# Visualise similarity for a movie
movie_title = 'Inception'
title_index = {t: i for i, t in enumerate(df['title'])}
idx = title_index[movie_title]

top_similar = sorted(enumerate(similarity[idx]), key=lambda x: x[1], reverse=True)[1:11]
sim_titles = [df.iloc[i]['title'] for i, _ in top_similar]
sim_scores = [round(s, 3) for _, s in top_similar]

plt.figure(figsize=(10, 5))
plt.barh(sim_titles[::-1], sim_scores[::-1], color='#e8b84b')
plt.title(f'Top 10 Movies Similar to "{movie_title}"')
plt.xlabel('Cosine Similarity')
plt.tight_layout()
plt.show()

## 6. TF-IDF vs Count Vectorizer Comparison

In [ ]:
# Compare both approaches
tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
tfidf_vectors = tfidf.fit_transform(df['tags']).toarray()
tfidf_similarity = cosine_similarity(tfidf_vectors)

def recommend_tfidf(title, n=5):
    title_index = {t: i for i, t in enumerate(df['title'])}
    if title not in title_index:
        return []
    idx = title_index[title]
    distances = sorted(enumerate(tfidf_similarity[idx]), key=lambda x: x[1], reverse=True)
    return [df.iloc[i]['title'] for i, _ in distances[1:n+1]]

test_movie = 'Avatar'
print(f'CountVec results for "{test_movie}":', recommend(test_movie))
print(f'TF-IDF results for "{test_movie}":', recommend_tfidf(test_movie))

## 7. Collaborative Filtering with SVD

> Note: The TMDB dataset doesn't include user-item ratings. For collaborative filtering,
> use the **MovieLens** dataset: https://grouplens.org/datasets/movielens/
> Download `ml-latest-small.zip` and extract `ratings.csv`.

In [ ]:
# Collaborative filtering using scikit-surprise
# Install: pip install scikit-surprise

import os
if os.path.exists('../data/ratings.csv'):
    from surprise import SVD, Dataset, Reader, accuracy
    from surprise.model_selection import cross_validate, train_test_split

    ratings = pd.read_csv('../data/ratings.csv')
    print('Ratings shape:', ratings.shape)
    print(ratings.head())

    reader = Reader(rating_scale=(0.5, 5.0))
    data = Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']], reader)

    # Train/test split
    trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

    # Train SVD
    svd = SVD(n_factors=100, n_epochs=20, random_state=42)
    svd.fit(trainset)

    # Evaluate
    predictions = svd.test(testset)
    print(f'\nRMSE: {accuracy.rmse(predictions):.4f}')
    print(f'MAE:  {accuracy.mae(predictions):.4f}')

    # Cross-validate
    cv_results = cross_validate(SVD(), data, measures=['RMSE', 'MAE'], cv=5, verbose=True)
else:
    print('ratings.csv not found. Download MovieLens dataset to use collaborative filtering.')

In [ ]:
# Predict rating for a user-movie pair
if os.path.exists('../data/ratings.csv'):
    user_id = 1
    movie_id = 50  # Toy Story
    pred = svd.predict(user_id, movie_id)
    print(f'Predicted rating for user {user_id} on movie {movie_id}: {pred.est:.2f}')

## 8. Summary

We've built:
- ✅ Content-based recommender using CountVectorizer + Cosine Similarity
- ✅ Comparison with TF-IDF
- ✅ Collaborative filtering with SVD (MovieLens dataset)

Next steps:
- Hybrid model combining both
- Deploy with Flask (see `app/app.py`)
- Add user login for personalised recommendations